# WIA3003: Academic Project II
## Logistic Regression Baseline

This experiment tests whether a linear model using the eight original measurements can match the initial QCNN. This is separate from the earlier logistic readout, which used four quantum outputs.

Use the same grouped training and test records as the four existing baselines. Keep the configuration fixed before evaluation. Codex assisted with writing and verifying this notebook.

## 1.0 Load Dataset and Saved Split

The file hash verifies that the saved row positions still refer to the same workbook.

In [1]:
import hashlib
import json
import platform
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

folder = Path.cwd()
project_folder = folder.parent if folder.name == "notebooks" else folder
reference_folder = project_folder / "results" / "qcnn_baseline"
details = json.loads((reference_folder / "run_details.json").read_text(encoding="utf-8"))

data_path = Path.home() / "Downloads" / "Orbital, Nasal and Craniofacial Class Pattern.xls"
dataset_hash = hashlib.sha256(data_path.read_bytes()).hexdigest()
if dataset_hash != details["dataset_sha256"]:
    raise ValueError("Workbook changed since the grouped QCNN run. Verify the data and row order first.")

df = pd.read_excel(data_path, sheet_name="Raw Data")
df.columns = df.columns.str.replace(r"\s+", " ", regex=True).str.strip()
features = details["features"]
X = df[features].apply(pd.to_numeric, errors="raise")
y = df["Gender"].astype("string").str.strip().map(details["label_mapping"])

assert len(features) == 8 and details["label_mapping"] == {"F": 0, "M": 1}
assert np.isfinite(X.to_numpy(dtype=float)).all() and not y.isna().any()
y = y.astype(int)

train_rows = np.asarray(details["train_row_positions"], dtype=int)
test_rows = np.asarray(details["test_row_positions"], dtype=int)
assert np.array_equal(np.sort(np.r_[train_rows, test_rows]), np.arange(len(X)))
groups = X.groupby(features, sort=False, dropna=False).ngroup()
assert not set(groups.iloc[train_rows]) & set(groups.iloc[test_rows])

X_train, X_test = X.iloc[train_rows], X.iloc[test_rows]
y_train, y_test = y.iloc[train_rows], y.iloc[test_rows]

reference_predictions = pd.read_csv(reference_folder / "test_predictions.csv")
assert np.array_equal(test_rows, reference_predictions["source_row_position"])
assert np.array_equal(y_test.to_numpy(), reference_predictions["actual_label"])
print("Training:", X_train.shape, "Testing:", X_test.shape)
print("Test row positions and labels match the saved QCNN predictions.")

Training: (832, 8) Testing: (210, 8)
Test row positions and labels match the saved QCNN predictions.


## 2.0 Train Logistic Regression

The pipeline learns scaling from the training records only. Logistic regression learns eight coefficients and one intercept. Use L2 regularisation with C=1 and no class weighting for this baseline. No hyperparameter search is performed.

In [2]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(solver="lbfgs", C=1.0, max_iter=1000)),
])

started = time.perf_counter()
model.fit(X_train, y_train)
fit_seconds = time.perf_counter() - started
logistic = model.named_steps["logistic"]
print("Fitted coefficients and intercept:", logistic.coef_.size + logistic.intercept_.size)
print("Solver iterations:", logistic.n_iter_.tolist())

Fitted coefficients and intercept: 9
Solver iterations: [12]


## 3.0 Evaluate on the Shared Test Set

Use the fixed 0.50 threshold. Binary precision, recall and F1 in the comparison table refer to the male class. The classification report also shows female results.

In [3]:
male_column = list(model.classes_).index(1)
probabilities = model.predict_proba(X_test)[:, male_column]
predictions = (probabilities >= 0.50).astype(int)

metrics = {
    "Accuracy": accuracy_score(y_test, predictions),
    "Precision (Male)": precision_score(y_test, predictions, zero_division=0),
    "Recall (Male)": recall_score(y_test, predictions, zero_division=0),
    "F1 (Male)": f1_score(y_test, predictions, zero_division=0),
    "Macro F1": f1_score(y_test, predictions, average="macro", zero_division=0),
    "ROC AUC": roc_auc_score(y_test, probabilities),
}
report = classification_report(y_test, predictions, labels=[0, 1],
    target_names=["Female", "Male"], output_dict=True, zero_division=0)
print(classification_report(y_test, predictions, labels=[0, 1],
    target_names=["Female", "Male"], digits=4, zero_division=0))
matrix = pd.DataFrame(confusion_matrix(y_test, predictions, labels=[0, 1]),
    index=["Actual Female", "Actual Male"], columns=["Predicted Female", "Predicted Male"])
display(matrix)

reference = pd.read_csv(reference_folder / "model_comparison.csv", index_col="Model")
comparison = pd.concat([reference,
    pd.DataFrame([metrics], index=pd.Index(["Logistic regression"], name="Model"))])
display(comparison.round(4))

              precision    recall  f1-score   support

      Female     0.8000    0.7391    0.7684        92
        Male     0.8080    0.8559    0.8313       118

    accuracy                         0.8048       210
   macro avg     0.8040    0.7975    0.7998       210
weighted avg     0.8045    0.8048    0.8037       210



,Predicted Female,Predicted Male
Actual Female,68,24
Actual Male,17,101


,Accuracy,Precision (Male),Recall (Male),F1 (Male),Macro F1,ROC AUC
Model,,,,,,
MLP,0.7143,0.7266,0.7881,0.7561,0.7056,0.7698
XGBoost,0.7905,0.7891,0.8559,0.8211,0.7841,0.8636
CNN,0.8000,0.7923,0.8729,0.8306,0.7932,0.8575
QCNN,0.6905,0.7154,0.7458,0.7303,0.6836,0.7792
Logistic regression,0.8048,0.8080,0.8559,0.8313,0.7998,0.8644


## 4.0 Save and Verify

Keep this baseline separate from earlier runs. Saved test predictions use workbook row positions so paired comparisons can align the same records.

In [4]:
output_folder = project_folder / "results" / "logistic_regression_baseline"
output_folder.mkdir(parents=True, exist_ok=True)
joblib.dump(model, output_folder / "logistic_pipeline.joblib")
comparison.to_csv(output_folder / "model_comparison.csv")
pd.DataFrame([metrics]).to_csv(output_folder / "metrics.csv", index=False)
matrix.to_csv(output_folder / "confusion_matrix.csv")
pd.DataFrame({
    "source_row_position": test_rows,
    "actual_label": y_test.to_numpy(),
    "predicted_label": predictions,
    "probability_male": probabilities,
}).to_csv(output_folder / "test_predictions.csv", index=False)
run_details = {
    "dataset_sha256": dataset_hash, "features": features,
    "label_mapping": details["label_mapping"],
    "train_row_positions": train_rows.tolist(), "test_row_positions": test_rows.tolist(),
    "solver": "lbfgs", "C": 1.0, "max_iter": 1000, "class_weight": None,
    "threshold": 0.5, "fit_seconds": fit_seconds,
    "python_version": platform.python_version(), "sklearn_version": sklearn.__version__,
    "metrics": metrics, "per_class_report": report,
}
(output_folder / "run_details.json").write_text(json.dumps(run_details, indent=2), encoding="utf-8")
reloaded = joblib.load(output_folder / "logistic_pipeline.joblib")
np.testing.assert_allclose(reloaded.predict_proba(X_test)[:, male_column], probabilities)
print("Saved pipeline reload reproduces the test probabilities.")

Saved pipeline reload reproduces the test probabilities.
